In [ ]:
%pip install gradio markdown python-docx

In [ ]:
import gradio as gr
import markdown
from docx import Document
from docx.shared import Pt, RGBColor
from html.parser import HTMLParser
import io

import tempfile
import os

def markdown_to_docx(md_text: str):
    doc = Document()

    for line in md_text.split("\n"):
        if line.startswith("### "):
            doc.add_heading(line[4:], level=3)
        elif line.startswith("## "):
            doc.add_heading(line[3:], level=2)
        elif line.startswith("# "):
            doc.add_heading(line[2:], level=1)
        elif line.startswith("- ") or line.startswith("* "):
            doc.add_paragraph(line[2:], style="List Bullet")
        elif line.strip() == "":
            doc.add_paragraph("")
        else:
            doc.add_paragraph(line)

    # Temporäre Datei im System-Temp-Verzeichnis anlegen (macOS-kompatibel)
    tmp = tempfile.NamedTemporaryFile(
        delete=False,
        suffix=".docx",
        prefix="markdown_output_"
    )
    doc.save(tmp.name)
    tmp.close()
    return tmp.name

# Gradio UI
with gr.Blocks(title="Markdown → Word") as demo:
    gr.Markdown("## 📄 Markdown zu Word Konverter")
    
    with gr.Row():
        with gr.Column():
            md_input = gr.Textbox(
                label="Markdown eingeben",
                lines=20,
                placeholder="# Überschrift\n\nText mit **fett** und *kursiv*..."
            )
            file_input = gr.File(label="Oder .md-Datei hochladen", file_types=[".md", ".markdown"])
        
        with gr.Column():
            preview = gr.Markdown(label="Live-Vorschau")
    
    download_btn = gr.Button("⬇️ Als Word herunterladen", variant="primary")
    output_file = gr.File(label="Download")

    # Live-Vorschau beim Tippen
    md_input.change(fn=lambda x: x, inputs=md_input, outputs=preview)

    # Datei einlesen
    def load_file(f):
        if f is None:
            return ""
        with open(f.name, "r", encoding="utf-8") as fh:
            return fh.read()
    file_input.change(fn=load_file, inputs=file_input, outputs=md_input)

    # Konvertierung
    download_btn.click(fn=markdown_to_docx, inputs=md_input, outputs=output_file)

demo.launch(inbrowser=True)